
Date: 2026-09-07
Tags: [[Programming language]], [[Python]], [[Algorithm]], [[Data Structure]], [[Recursion]], [[Mathematics]]

---

# Recursion Run Amok

> [!info] Definitions & Properties
> Recursive algorithms become inefficient when subproblems overlap or when calls branch repeatedly without reducing problem size. Infinite recursion occurs when recursive calls fail to reach a base case, ultimately consuming system resources and exhausting available call-stack memory.

---

## 1. Exponential Call Branching

Poorly designed recursion can cause an explosion in the number of function calls, turning simple linear problems into exponential-time algorithms.

### Element Uniqueness via Binary Branching

Section 3.3.3 introduced the element uniqueness problem. The implementation in Code Fragment 4.6 attempts to solve it by testing if the first $n - 1$ elements are unique, the last $n - 1$ elements are unique, and the first and last elements differ.

In [ ]:
def unique3(S, start, stop):
    """Return True if there are no duplicate elements in slice S[start:stop]."""
    if stop - start <= 1:
        return True                             # at most one item
    elif not unique3(S, start, stop - 1):
        return False                            # first part has duplicate
    elif not unique3(S, start + 1, stop):
        return False                            # second part has duplicate
    else:
        return S[start] != S[stop - 1]          # do first and last differ?


> [!warning] Inefficiency of `unique3`
> A problem of size $n$ triggers two subproblems of size $n - 1$, which trigger four subproblems of size $n - 2$, and so on. The total number of invocations follows a geometric progression (see Proposition 3.5):
> $$1 + 2 + 4 + \dots + 2^{n-1} = 2^n - 1$$
> 
> 
> Since each call takes $O(1)$ nonrecursive work, the overall running time is $O(2^n)$.

---

### Inefficient Fibonacci Numbers

The mathematical definition of the $n$-th Fibonacci number is:

$$\begin{aligned} F_0 &= 0 \\ F_1 &= 1 \\ F_n &= F_{n-2} + F_{n-1} \quad \text{for } n > 1 \end{aligned}$$

Directly translating this formula into a binary recursive function causes massive recalculation of identical subproblems (see Code Fragment 4.7).

In [ ]:
def bad_fibonacci(n):
    """Return the nth Fibonacci number."""
    if n <= 1:
        return n
    else:
        return bad_fibonacci(n - 2) + bad_fibonacci(n - 1)


Let $c_n$ denote the total number of calls required to compute `bad_fibonacci(n)`:

| $n$ | Recursive Expansion of Call Count $c_n$ | Total Call Invocations ($c_n$) |
| --- | --- | --- |
| **0** | Base case ($n = 0$) | $1$ |
| **1** | Base case ($n = 1$) | $1$ |
| **2** | $1 + c_0 + c_1 = 1 + 1 + 1$ | $3$ |
| **3** | $1 + c_1 + c_2 = 1 + 1 + 3$ | $5$ |
| **4** | $1 + c_2 + c_3 = 1 + 3 + 5$ | $9$ |
| **5** | $1 + c_3 + c_4 = 1 + 5 + 9$ | $15$ |
| **6** | $1 + c_4 + c_5 = 1 + 9 + 15$ | $25$ |
| **7** | $1 + c_5 + c_6 = 1 + 15 + 25$ | $41$ |
| **8** | $1 + c_6 + c_7 = 1 + 25 + 41$ | $67$ |

The number of calls more than doubles every two indices ($c_4 > 2c_2$, $c_6 > 2c_4$). In general:

$$c_n > 2^{n/2}$$

The overall running time is exponential in $n$, making `bad_fibonacci` unusable even for moderate values like $n = 50$.

---

## 2. Linear Recursion Optimization

To compute Fibonacci numbers efficiently, avoid duplicating work by returning a pair of consecutive values $(F_n, F_{n-1})$, using the convention $F_{-1} = 0$.

In [ ]:
def good_fibonacci(n):
    """Return pair of Fibonacci numbers, (F(n), F(n-1))."""
    if n <= 1:
        return (n, 0)
    else:
        (a, b) = good_fibonacci(n - 1)
        return (a + b, a)

print(good_fibonacci(8))

| Metric | `bad_fibonacci(n)` | `good_fibonacci(n)` |
| --- | --- | --- |
| **Recursion Pattern** | Binary recursion (two recursive calls per non-base invocation) | Linear recursion (one recursive call per invocation) |
| **Total Invocations** | Exponential: $c_n > 2^{n/2}$ | Linear: $n$ calls |
| **Work per Call** | $O(1)$ | $O(1)$ |
| **Time Complexity** | $O(2^n)$ | $O(n)$ |
| **Space (Call Stack)** | $O(n)$ activation frames | $O(n)$ activation frames |

---

## 3. Maximum Recursive Depth in Python

Section 4.3.1 examines infinite recursion. An infinite recursion occurs when recursive calls fail to make progress toward a base case.

In [ ]:
def fib(n):
    return fib(n)  # infinite recursion: no base case


Subtle bugs also cause infinite recursion. In binary search (see Section 4.1.3, Code Fragment 4.3), if the right-half recursive call is erroneously written as:

In [ ]:
return binary_search(data, target, mid, high)  # incorrect: should use mid + 1


Searching a two-element range can result in a recursive call on the exact same interval, triggering infinite recursion.

### Call Stack Limits and Reconfiguration

To prevent unbounded stack growth from exhausting system memory, Python sets an artificial limit on the number of simultaneous active activation records (typically 1000). Exceeding this limit causes the interpreter to raise an exception:

```text
RuntimeError: maximum recursion depth exceeded

```

* Logarithmic algorithms (e.g., `binary_search`) have a recursive depth of $O(\log n)$ and never hit this limit during practical use ($2^{1000}$ exceeds the number of atoms in the universe).
* Linear recursive algorithms with depth proportional to $n$ will hit the 1000-call limit on modest inputs.

The recursion limit can be inspected and updated dynamically using the `sys` module:

In [ ]:
import sys

old_limit = sys.getrecursionlimit()     # returns current limit (e.g., 1000)
sys.setrecursionlimit(1000000)          # reconfigures limit to 1,000,000 frames


---
# ~={red}References=~
- [[Data_Structures_and_Algorithms_in_Python_Michael_T_Goodrich_z_library.pdf]](p, 165)